# SCARA: forward and inverse dynamics

This notebook presents the Euler–Lagrange dynamics of our four-axis SCARA, following `Handwritten-stuff/Forward_Inverse_Dynamics.pdf`. It builds the COM Jacobians, the inertia matrix $D(q)$, the Coriolis matrix $C(q,\dot q)$ and the gravity vector $G(q)$. It then uses them for inverse dynamics (torques for a planned motion) and forward dynamics (motion caused by applied torques). Each result is compared with MuJoCo's own rigid-body algorithms. The final cell opens **one native `mujoco.viewer` window**. In it you set joint torques and watch the robot move, with our model's prediction drawn as a ghost on top.

$$D(q)\ddot q + C(q,\dot q)\dot q + G(q) = \tau,\qquad \tau=[\tau_1,\tau_2,F_3,\tau_4]^T$$

![Our SCARA model](../figures/model_preview.png)

**Run locally:** open this file in VS Code with its Jupyter support, select the project's `.venv/Scripts/python.exe` kernel, and run the cells in order. On a fresh setup, install the project requirements plus `notebooks/requirements.txt`. The viewer opens as a desktop window, so a hosted or headless notebook service can't display it. The final cell stays busy until you close that window.

In [1]:
from pathlib import Path
from collections import deque
from queue import SimpleQueue, Empty
import json
import math
import sys
import time

import numpy as np
import mujoco
import mujoco.viewer

# Works when the notebook is opened from the repository or notebooks folder.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "config" / "robot.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the SCARA project folder.")
sys.path.insert(0, str(ROOT / "scripts" / "dynamics"))
from scara_dynamics import (
    DYNAMICS, mass_matrix, coriolis_matrix, gravity_vector, inverse_dynamics,
    forward_dynamics, kinetic_energy, potential_energy, rk4_step,
)
from scara_kinematics import forward_kinematics
from view_dynamics import (load_arm_model, desired, GRAVITY_HOLD,
                           Q0_FORWARD, AMPLITUDE, OMEGA, Q0_INVERSE)

config = json.loads((ROOT / "config" / "robot.json").read_text())
np.set_printoptions(precision=5, suppress=True)
print("MuJoCo:", mujoco.__version__)
print("Project:", ROOT)

MuJoCo: 3.17.0
Project: E:\sem5\robotics\SCARA-RobotManipulator


## Model and parameters

The joint order is $q=[\theta_1,\theta_2,q_3,\theta_4]$: shoulder, elbow, downward slide and downward-axis wrist. The DH slide distance is $d_3=d_{3,\min}+q_3$, so $\dot d_3=\dot q_3$. The derivation treats the robot as four rigid bodies:

| Body | Moves with | COM |
| --- | --- | --- |
| 1 upper arm | $\theta_1$ | $r_1$ along link 1 |
| 2 forearm | $\theta_1+\theta_2$ | $r_2$ along link 2 |
| 3 vertical carriage | $\theta_1+\theta_2$, slides with $d_3$ | on the vertical axis |
| 4 wrist + palm + fingers | yaw $\theta_1+\theta_2-\theta_4$, slides with $d_3$ | on the vertical axis |

Every body rotates about a vertical axis, so only each body's $I_{zz}$ about its COM matters. The parameters are read from `models/scara.xml`. The two fingers are held closed and counted as part of body 4. They sit off the wrist axis, so they add $m_f y_f^2$ to $I_4$ (the parallel-axis theorem).

For the comparisons, `load_arm_model()` loads the same MJCF with three changes: it welds the fingers to the wrist, turns off joint damping (our model has no friction), and uses MuJoCo's RK4 integrator.

In [2]:
p = DYNAMICS
print(f"a1 = {p.a1:.3f} m   a2 = {p.a2:.3f} m   r1 = {p.r1:.3f} m   r2 = {p.r2:.3f} m")
print(f"m1 = {p.m1:.3f} kg  m2 = {p.m2:.3f} kg  m3 = {p.m3:.3f} kg  m4 = {p.m4:.3f} kg  mu = m3+m4 = {p.mu:.3f} kg")
print(f"I1 = {p.I1:.2e}  I2 = {p.I2:.2e}  I3 = {p.I3:.2e}  I4 = {p.I4:.2e} kg m^2   g = {p.g} m/s^2")

model = load_arm_model(damping=False)
data = mujoco.MjData(model)
print("Comparison model: nq =", model.nq, "| actuators:", [model.actuator(i).name or model.joint(model.actuator_trnid[i, 0]).name for i in range(model.nu)])

q_example = np.array([math.radians(30), math.radians(-70), .045, math.radians(25)])
qd_example = np.array([.8, -1.2, .05, 2.0])       # rad/s, rad/s, m/s, rad/s
qdd_example = np.array([1.5, -2.0, .3, 4.0])      # rad/s^2, rad/s^2, m/s^2, rad/s^2

def set_state(q, qd=np.zeros(4), qdd=np.zeros(4)):
    data.qpos[:], data.qvel[:] = q, qd
    mujoco.mj_forward(model, data)
    data.qacc[:] = qdd   # set after mj_forward, which overwrites qacc

a1 = 0.240 m   a2 = 0.180 m   r1 = 0.120 m   r2 = 0.090 m
m1 = 1.100 kg  m2 = 0.750 kg  m3 = 0.300 kg  m4 = 0.300 kg  mu = m3+m4 = 0.600 kg
I1 = 6.00e-03  I2 = 2.50e-03  I3 = 1.00e-04  I4 = 5.23e-04 kg m^2   g = 9.81 m/s^2
Comparison model: nq = 4 | actuators: ['shoulder', 'elbow', 'vertical', 'wrist_roll']


## 1. COM Jacobians and kinetic energy

Each body's COM velocity and angular velocity are linear in $\dot q$: $v_{c_i}=J_{v_i}(q)\dot q$ and $\omega_i=J_{\omega_i}\dot q$. With $s_{12}=\sin(\theta_1+\theta_2)$ and $c_{12}=\cos(\theta_1+\theta_2)$:

$$
J_{v_1}=\begin{bmatrix}-r_1s_1&0&0&0\\r_1c_1&0&0&0\\0&0&0&0\end{bmatrix},\quad
J_{v_2}=\begin{bmatrix}-a_1s_1-r_2s_{12}&-r_2s_{12}&0&0\\a_1c_1+r_2c_{12}&r_2c_{12}&0&0\\0&0&0&0\end{bmatrix},
$$

$$
J_{v_3}=J_{v_4}=\begin{bmatrix}-a_1s_1-a_2s_{12}&-a_2s_{12}&0&0\\a_1c_1+a_2c_{12}&a_2c_{12}&0&0\\0&0&-1&0\end{bmatrix}.
$$

The third row of $J_{v_3}$ and $J_{v_4}$ carries $\dot d_3$; $d_3$ points down, so $\dot z=-\dot d_3$. The angular velocities are about $z$ only:

$$\omega_1=\dot\theta_1,\quad \omega_2=\omega_3=\dot\theta_1+\dot\theta_2,\quad \omega_4=\dot\theta_1+\dot\theta_2-\dot\theta_4.$$

The kinetic energy is $K=\sum_i\left(\tfrac12 m_i v_{c_i}^Tv_{c_i}+\tfrac12 I_i\omega_i^2\right)=\tfrac12\dot q^TD(q)\dot q$, so

$$D(q)=\sum_{i=1}^4\left(m_iJ_{v_i}^TJ_{v_i}+I_iJ_{\omega_i}^TJ_{\omega_i}\right).$$

The cell below builds these Jacobians and compares them with MuJoCo's `mj_jacBodyCom`.

In [3]:
def com_jacobians(q, p=DYNAMICS):
    """Linear COM Jacobians (3x4) and z-angular Jacobian rows (1x4) of bodies 1-4."""
    q1, q2 = q[0], q[1]
    s1, c1, s12, c12 = math.sin(q1), math.cos(q1), math.sin(q1 + q2), math.cos(q1 + q2)
    Jv1 = np.array([[-p.r1*s1, 0, 0, 0], [p.r1*c1, 0, 0, 0], [0, 0, 0, 0]])
    Jv2 = np.array([[-p.a1*s1 - p.r2*s12, -p.r2*s12, 0, 0],
                    [ p.a1*c1 + p.r2*c12,  p.r2*c12, 0, 0], [0, 0, 0, 0]])
    Jv3 = np.array([[-p.a1*s1 - p.a2*s12, -p.a2*s12, 0, 0],
                    [ p.a1*c1 + p.a2*c12,  p.a2*c12, 0, 0], [0, 0, -1, 0]])
    Jw = [np.array([1, 0, 0, 0]), np.array([1, 1, 0, 0]),
          np.array([1, 1, 0, 0]), np.array([1, 1, 0, -1])]
    return [Jv1, Jv2, Jv3, Jv3.copy()], Jw

Jv, Jw = com_jacobians(q_example)
set_state(q_example)
for i, body in enumerate(["upper_arm", "forearm", "vertical_carriage", "wrist"]):
    jacp, jacr = np.zeros((3, model.nv)), np.zeros((3, model.nv))
    mujoco.mj_jacBodyCom(model, data, jacp, jacr, model.body(body).id)
    assert np.allclose(Jv[i], jacp, atol=1e-12) and np.allclose(Jw[i], jacr[2], atol=1e-12)
    assert np.allclose(jacr[:2], 0, atol=1e-12)   # no tilting: rotation is about z only
print("COM Jacobians J_v1..J_v4 and J_w1..J_w4 match MuJoCo: PASS")

masses = [p.m1, p.m2, p.m3, p.m4]
inertias = [p.I1, p.I2, p.I3, p.I4]
D_from_jacobians = sum(m * J.T @ J + I * np.outer(w, w)
                       for m, I, J, w in zip(masses, inertias, Jv, Jw))
assert np.allclose(D_from_jacobians, mass_matrix(q_example), atol=1e-14)
print("Sum of m J_v^T J_v + I J_w^T J_w equals the closed-form D(q): PASS")

COM Jacobians J_v1..J_v4 and J_w1..J_w4 match MuJoCo: PASS
Sum of m J_v^T J_v + I J_w^T J_w equals the closed-form D(q): PASS


## 2. Inertia matrix $D(q)$

Expanding the sum gives

$$
D(q)=\begin{bmatrix}d_{11}&d_{12}&0&-I_4\\d_{12}&d_{22}&0&-I_4\\0&0&\mu&0\\-I_4&-I_4&0&I_4\end{bmatrix},\qquad \mu=m_3+m_4,
$$

$$
\begin{aligned}
d_{11}&=m_1r_1^2+I_1+m_2(a_1^2+r_2^2+2a_1r_2c_2)+\mu(a_1^2+a_2^2+2a_1a_2c_2)+I_2+I_3+I_4,\\
d_{12}&=m_2(r_2^2+a_1r_2c_2)+\mu(a_2^2+a_1a_2c_2)+I_2+I_3+I_4,\\
d_{22}&=m_2r_2^2+\mu a_2^2+I_2+I_3+I_4.
\end{aligned}
$$

Only $\theta_2$ appears, through $c_2=\cos\theta_2$. The shoulder angle can't change the inertia because the whole arm just rotates about the base. The slide is decoupled from the rotations: its only inertia is the moving mass $\mu$. The $-I_4$ terms couple the wrist to the arm. With no wrist torque, the wrist keeps its yaw in the world while the arm swings. MuJoCo forms the same matrix internally (`mj_fullM`).

In [4]:
D = mass_matrix(q_example)
set_state(q_example)
D_mujoco = np.zeros((model.nv, model.nv))
mujoco.mj_fullM(model, data, D_mujoco)
print("D(q) at the example pose:\n", D)
print("Eigenvalues:", np.linalg.eigvalsh(D), "(all positive: D is positive definite)")
print("max |D - MuJoCo| =", np.abs(D - D_mujoco).max())
assert np.allclose(D, D_mujoco, atol=1e-12)

for q2_deg in (0, 45, 90, 135):
    d11 = mass_matrix([0, math.radians(q2_deg), 0, 0])[0, 0]
    print(f"q2 = {q2_deg:3d} deg -> shoulder inertia d11 = {d11:.4f} kg m^2")

D(q) at the example pose:
 [[ 0.15705  0.04304  0.      -0.00052]
 [ 0.04304  0.02864  0.      -0.00052]
 [ 0.       0.       0.6      0.     ]
 [-0.00052 -0.00052  0.       0.00052]]
Eigenvalues: [0.00051 0.01555 0.17015 0.6    ] (all positive: D is positive definite)
max |D - MuJoCo| = 5.551115123125783e-17
q2 =   0 deg -> shoulder inertia d11 = 0.2125 kg m^2
q2 =  45 deg -> shoulder inertia d11 = 0.1878 kg m^2
q2 =  90 deg -> shoulder inertia d11 = 0.1282 kg m^2
q2 = 135 deg -> shoulder inertia d11 = 0.0687 kg m^2


## 3. Coriolis/centrifugal and gravity terms

The Coriolis matrix comes from the Christoffel symbols of $D$:

$$
c_{ijk}=\tfrac12\left(\frac{\partial d_{kj}}{\partial q_i}+\frac{\partial d_{ki}}{\partial q_j}-\frac{\partial d_{ij}}{\partial q_k}\right),\qquad C_{kj}=\sum_i c_{ijk}\,\dot q_i.
$$

Only $d_{11}$ and $d_{12}$ depend on $q$ (through $\theta_2$). Let $h=(m_2a_1r_2+\mu a_1a_2)\sin\theta_2$. Then

$$
C(q,\dot q)=\begin{bmatrix}-h\dot\theta_2&-h(\dot\theta_1+\dot\theta_2)&0&0\\h\dot\theta_1&0&0&0\\0&0&0&0\\0&0&0&0\end{bmatrix},\qquad
C\dot q=\begin{bmatrix}-h(2\dot\theta_1\dot\theta_2+\dot\theta_2^2)\\h\dot\theta_1^2\\0\\0\end{bmatrix}.
$$

The potential energy is $P=\sum m_i g z_{c_i}=-\mu g d_3+\text{const}$, because only bodies 3 and 4 change height. So $G=\partial P/\partial q=[0,0,-\mu g,0]^T$. To hold the carriage still, the slide actuator must supply $F_3=-\mu g$, an upward push.

The cell below checks $C$ against the Christoffel formula (numerical derivatives of $D$), checks that $\dot D-2C$ is skew-symmetric, and compares $C\dot q+G$ with MuJoCo's bias force `qfrc_bias`.

In [5]:
def dD_dq(q, step=1e-6):
    return np.array([(mass_matrix(q + step*e) - mass_matrix(q - step*e)) / (2*step) for e in np.eye(4)])

dD = dD_dq(q_example)
C_christoffel = np.array([[sum(.5*(dD[i][k, j] + dD[j][k, i] - dD[k][i, j]) * qd_example[i] for i in range(4))
                           for j in range(4)] for k in range(4)])
C = coriolis_matrix(q_example, qd_example)
print("C(q, qdot):\n", C)
assert np.allclose(C, C_christoffel, atol=1e-8)
print("C equals the Christoffel-symbol construction: PASS")

N = np.einsum("kij,k->ij", dD, qd_example) - 2*C
assert np.allclose(N, -N.T, atol=1e-8)
print("D_dot - 2C is skew-symmetric: PASS")

set_state(q_example, qd_example)
bias = C @ qd_example + gravity_vector(q_example)
print("C qdot + G       :", bias)
print("MuJoCo qfrc_bias :", data.qfrc_bias)
assert np.allclose(bias, data.qfrc_bias, atol=1e-12)
print(f"Holding force on the slide: F3 = -mu g = {-p.mu*p.g:.3f} N")

C(q, qdot):
 [[-0.0475  -0.01583  0.       0.     ]
 [-0.03166  0.       0.       0.     ]
 [ 0.       0.       0.       0.     ]
 [ 0.       0.       0.       0.     ]]
C equals the Christoffel-symbol construction: PASS
D_dot - 2C is skew-symmetric: PASS
C qdot + G       : [-0.019   -0.02533 -5.886    0.     ]
MuJoCo qfrc_bias : [-0.019   -0.02533 -5.886    0.     ]
Holding force on the slide: F3 = -mu g = -5.886 N


## 4. Inverse dynamics

Inverse dynamics answers this question: *what actuator efforts produce a chosen motion?* It needs no matrix inversion:

$$\tau = D(q)\ddot q + C(q,\dot q)\dot q + G(q).$$

The test motion is $q_d(t)=q_0+B\,(1-\cos\omega t)$ for each joint. It starts at rest and stays inside the joint limits. The cell evaluates our $\tau(t)$, compares it with MuJoCo's recursive Newton–Euler result (`mj_inverse`), and lists the peak efforts against the actuator limits in `config/robot.json`.

In [6]:
times = np.arange(0, 4.0 + 1e-9, .01)
taus = np.array([inverse_dynamics(*desired(t)) for t in times])

worst = 0.0
for t, tau in zip(times, taus):
    q, qd, qdd = desired(t)
    set_state(q, qd, qdd)
    mujoco.mj_inverse(model, data)
    worst = max(worst, np.abs(tau - data.qfrc_inverse).max())
assert worst < 1e-10
print(f"max |tau - MuJoCo mj_inverse| over 4 s: {worst:.1e}\n")

print(f"{'t (s)':>6} | {'tau1 (Nm)':>10} {'tau2 (Nm)':>10} {'F3 (N)':>10} {'tau4 (Nm)':>10}")
for t, tau in zip(times[::50], taus[::50]):
    print(f"{t:6.2f} | " + " ".join(f"{x:10.4f}" for x in tau))

limits = config["effort_limit"]
limits = [limits["shoulder_Nm"], limits["elbow_Nm"], limits["vertical_N"], limits["wrist_Nm"]]
print()
for name, peak, limit in zip(["shoulder", "elbow", "vertical", "wrist"], np.abs(taus).max(axis=0), limits):
    print(f"{name:9s} peak |effort| = {peak:7.4f}  (limit {limit}, {100*peak/limit:5.1f} %)")

max |tau - MuJoCo mj_inverse| over 4 s: 1.4e-16

 t (s) |  tau1 (Nm)  tau2 (Nm)     F3 (N)  tau4 (Nm)
  0.00 |     0.0855     0.0103    -5.7780     0.0043
  0.50 |     0.0951     0.0258    -5.8276     0.0014
  1.00 |     0.0946     0.0333    -5.9309    -0.0035
  1.50 |     0.0392     0.0159    -5.9929    -0.0039
  2.00 |    -0.0333    -0.0080    -5.9566     0.0006
  2.50 |    -0.1477    -0.0510    -5.8554     0.0040
  3.00 |    -0.2389    -0.0818    -5.7823     0.0022
  3.50 |    -0.1920    -0.0634    -5.8046    -0.0019
  4.00 |    -0.1002    -0.0194    -5.9017    -0.0026

shoulder  peak |effort| =  0.2396  (limit 12,   2.0 %)
elbow     peak |effort| =  0.0821  (limit 8,   1.0 %)
vertical  peak |effort| =  5.9940  (limit 80,   7.5 %)
wrist     peak |effort| =  0.0045  (limit 2,   0.2 %)


## 5. Forward dynamics

Forward dynamics answers the opposite question: *how does the robot move under given efforts?* We solve the linear system

$$D(q)\,\ddot q = \tau - C(q,\dot q)\dot q - G(q)$$

for $\ddot q$ and integrate it in time (RK4, 1 ms steps). Both simulations start from the same state and receive the same torques: $\tau=A\cos\omega t$ on each joint, plus $F_3=-\mu g$ to hold up the carriage. One is our equations, the other is MuJoCo. If the derivation is right, the two trajectories should agree to rounding error.

As a second check, we hold the joints free of external work: zero torques and only the gravity hold on the slide. MuJoCo then moves the robot, and our $K=\tfrac12\dot q^TD(q)\dot q$, evaluated on MuJoCo's states, should stay constant.

In [7]:
dt = model.opt.timestep
def torque_profile(t):
    return GRAVITY_HOLD + AMPLITUDE * np.cos(OMEGA * t)

mujoco.mj_resetData(model, data)
data.qpos[:] = Q0_FORWARD
q_ours, qd_ours = Q0_FORWARD.copy(), np.zeros(4)
worst, samples = 0.0, []
for k in range(4000):
    tau = torque_profile(k * dt)
    data.ctrl[:] = tau
    mujoco.mj_step(model, data)
    q_ours, qd_ours = rk4_step(q_ours, qd_ours, tau, dt)
    assert data.nefc == 0, "a joint limit became active"
    worst = max(worst, np.abs(data.qpos - q_ours).max())
    if (k + 1) % 1000 == 0:
        samples.append(((k + 1) * dt, data.qpos.copy(), q_ours.copy()))
for t, q_mj, q_us in samples:
    print(f"t = {t:.1f} s  MuJoCo q = {q_mj}  ours q = {q_us}")
print(f"max |q_MuJoCo - q_ours| over 4 s: {worst:.1e}")
assert worst < 1e-9

# Energy check: no actuator work (gravity is cancelled by the hold force).
model.opt.enableflags |= mujoco.mjtEnableBit.mjENBL_ENERGY
mujoco.mj_resetData(model, data)
data.qpos[:] = [-.3, 1.0, .05, 0]
data.qvel[:] = [1.5, -2.0, 0, -1.0]
data.ctrl[:] = GRAVITY_HOLD
energies = []
for k in range(2000):
    mujoco.mj_step(model, data)
    mujoco.mj_forward(model, data)   # refresh data.energy for the new state
    assert data.nefc == 0, "a joint limit became active"
    energies.append((kinetic_energy(data.qpos, data.qvel), data.energy[1]))
energies = np.array(energies)
print(f"\nK from our D(q): start {energies[0, 0]:.6f} J, spread over 2 s {np.ptp(energies[:, 0]):.1e} J")
print(f"max |K_ours - K_MuJoCo| = {np.abs(energies[:, 0] - energies[:, 1]).max():.1e} J")
assert np.ptp(energies[:, 0]) < 1e-9 and np.allclose(energies[:, 0], energies[:, 1], atol=1e-12)
model.opt.enableflags &= ~int(mujoco.mjtEnableBit.mjENBL_ENERGY)

t = 1.0 s  MuJoCo q = [ 0.1037  -1.10575  0.10464 -0.99823]  ours q = [ 0.1037  -1.10575  0.10464 -0.99823]
t = 2.0 s  MuJoCo q = [ 0.4758  -1.83164  0.02042 -1.34819]  ours q = [ 0.4758  -1.83164  0.02042 -1.34819]
t = 3.0 s  MuJoCo q = [ 0.26434 -1.97782  0.10506 -1.70201]  ours q = [ 0.26434 -1.97782  0.10506 -1.70201]
t = 4.0 s  MuJoCo q = [-0.07952 -0.86965  0.02083 -0.93388]  ours q = [-0.07952 -0.86965  0.02083 -0.93388]
max |q_MuJoCo - q_ours| over 4 s: 3.1e-14

K from our D(q): start 0.098562 J, spread over 2 s 8.8e-13 J
max |K_ours - K_MuJoCo| = 2.6e-16 J


## 6. Interactive dynamics viewer

The final cell opens the native MuJoCo viewer. MuJoCo simulates the robot. At the same time, our equations are integrated from the same state with the same efforts. Their prediction is drawn as **green ghost spheres** at the elbow and tool. If the derivation is correct, the ghosts stay on the robot.

| Mode | What happens |
| --- | --- |
| **TORQUE / forward dynamics** | You set a constant effort for each joint. The robot accelerates, and the ghost shows what our $\ddot q=D^{-1}(\tau-C\dot q-G)$ predicts. The slide gets $-\mu g$ added automatically unless you switch the gravity hold off. |
| **TRAJECTORY / inverse dynamics** | The planned motion $q_d(t)$ from section 4 is shown as the ghost. Our $\tau=D\ddot q_d+C\dot q_d+G$ is sent to MuJoCo open loop (no feedback), and the robot follows it. |

**Keys:** F8 switches mode. F9 selects a joint. Left and Right decrease or increase its effort. F10 zeroes all efforts. Down toggles the slide's gravity hold. F11 runs or pauses. F12 resets.

A constant torque keeps accelerating a joint until it reaches its limit. MuJoCo then applies a limit force that our 4-DOF model doesn't have. When that happens, the overlay says so and the ghost is re-synced to the robot. Start with small efforts, such as 0.1–0.3 Nm on the shoulder.

In [8]:
class DynamicsDemo:
    """MuJoCo simulation plus our equations, driven by viewer key events."""
    MODES = ["TORQUE / forward dynamics", "TRAJECTORY / inverse dynamics"]
    STEPS = np.array([.05, .02, .5, .002])        # effort change per key press
    UNITS = ["Nm", "Nm", "N", "Nm"]
    NAMES = ["Shoulder", "Elbow", "Slide", "Wrist"]
    PERIOD = 6.0                                   # trajectory replay length (s)

    def __init__(self):
        self.model = load_arm_model(damping=False)
        self.data = mujoco.MjData(self.model)
        self.dt = self.model.opt.timestep
        self.ctrl_range = self.model.actuator_ctrlrange.copy()
        self.commands = SimpleQueue()
        self.trail = deque(maxlen=400)
        self.mode = 0
        self.selected = 0
        self.user_tau = np.zeros(4)
        self.gravity_hold = True
        self.reset()

    def reset(self):
        self.running = False
        self.t = 0.0
        self.limit_active = False
        self.error = 0.0
        self.status = "Ready: F11 to run"
        q0 = Q0_FORWARD if self.mode == 0 else Q0_INVERSE
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] = q0
        mujoco.mj_forward(self.model, self.data)
        self.q_ref, self.qd_ref = q0.copy(), np.zeros(4)
        self.tau = self.torque()
        self.trail.clear()

    def torque(self):
        if self.mode == 1:
            return inverse_dynamics(*desired(self.t + self.dt / 2))
        return self.user_tau + (GRAVITY_HOLD if self.gravity_hold else 0)

    def on_key(self, keycode):
        # The callback only queues an action; the main loop applies it under viewer.lock().
        names = {297: "mode", 298: "select", 263: "minus", 262: "plus",
                 299: "zero", 264: "gravity", 300: "run", 301: "reset"}
        if keycode in names:
            self.commands.put(names[keycode])

    def handle(self, action):
        if action == "mode":
            self.mode = 1 - self.mode
            self.reset()
        elif action == "select":
            self.selected = (self.selected + 1) % 4
        elif action in ("minus", "plus") and self.mode == 0:
            i = self.selected
            hold = GRAVITY_HOLD[i] if self.gravity_hold else 0
            value = self.user_tau[i] + (1 if action == "plus" else -1) * self.STEPS[i]
            self.user_tau[i] = np.clip(round(value / self.STEPS[i]) * self.STEPS[i],
                                       self.ctrl_range[i, 0] - hold, self.ctrl_range[i, 1] - hold)
        elif action == "zero":
            self.user_tau[:] = 0
        elif action == "gravity" and self.mode == 0:
            self.gravity_hold = not self.gravity_hold
        elif action == "run":
            self.running = not self.running
            self.status = "Running" if self.running else "Paused"
        elif action == "reset":
            self.reset()

    def step(self):
        self.tau = np.clip(self.torque(), self.ctrl_range[:, 0], self.ctrl_range[:, 1])
        self.data.ctrl[:] = self.tau
        mujoco.mj_step(self.model, self.data)
        self.t += self.dt
        if self.mode == 0:
            self.q_ref, self.qd_ref = rk4_step(self.q_ref, self.qd_ref, self.tau, self.dt)
        else:
            self.q_ref, self.qd_ref, _ = desired(self.t)
        self.limit_active = self.data.nefc > 0
        if self.limit_active and self.mode == 0:
            # MuJoCo's limit force is outside our model: restart the comparison from here.
            self.q_ref, self.qd_ref = self.data.qpos.copy(), self.data.qvel.copy()
        self.error = float(np.abs(self.data.qpos - self.q_ref).max())

    def tick(self, wall_dt):
        if not self.running:
            return
        for _ in range(min(int(round(wall_dt / self.dt)), 40)):
            self.step()
        if self.mode == 1 and self.t >= self.PERIOD:
            self.reset()
            self.running = True
        self.status = ("Joint limit: MuJoCo limit force (not in our model), ghost re-synced"
                       if self.limit_active else "Running")
        tool = self.data.site("tool").xpos.copy()
        if not self.trail or np.linalg.norm(tool - self.trail[-1]) > .002:
            self.trail.append(tool)

    def ghost_points(self):
        q = self.q_ref
        elbow = np.array([p.a1 * math.cos(q[0]), p.a1 * math.sin(q[0]), config["base_height_m"]])
        return elbow, forward_kinematics(q)[:3]

    def overlay(self):
        scale = np.array([180 / math.pi, 180 / math.pi, 1000, 180 / math.pi])
        lines = ["SCARA | " + self.MODES[self.mode], f"t = {self.t:5.2f} s", ""]
        if self.mode == 0:
            lines.append("Set efforts (gravity hold " + ("ON" if self.gravity_hold else "OFF") + "):")
            for i in range(4):
                lines.append(f"{'>' if i == self.selected else ' '} {self.NAMES[i]}: "
                             f"{self.user_tau[i]: .3f} {self.UNITS[i]}")
            lines.append("")
        lines.append("Applied tau: " + ", ".join(f"{x:.3f}" for x in self.tau))
        lines += ["", "q MuJoCo (deg, mm): " + ", ".join(f"{x:.1f}" for x in self.data.qpos * scale),
                  ("q ours    " if self.mode == 0 else "q planned ") + "(deg, mm): "
                  + ", ".join(f"{x:.1f}" for x in self.q_ref * scale),
                  f"max |difference| = {self.error:.1e}",
                  f"K = 1/2 qd^T D qd = {kinetic_energy(self.data.qpos, self.data.qvel):.4f} J",
                  "", self.status]
        return "\n".join(lines)


def add_marker(scene, kind, start, size, color, end=None):
    if scene.ngeom >= scene.maxgeom:
        return
    geom = scene.geoms[scene.ngeom]
    mujoco.mjv_initGeom(geom, kind, np.asarray(size, dtype=float), np.asarray(start, dtype=float),
                        np.eye(3).ravel(), np.asarray(color, dtype=np.float32))
    if end is not None:
        mujoco.mjv_connector(geom, kind, size[0], np.asarray(start, dtype=float), np.asarray(end, dtype=float))
    scene.ngeom += 1


def update_markers(demo, scene):
    scene.ngeom = 0
    points = list(demo.trail)
    for a, b in zip(points, points[1:]):
        add_marker(scene, mujoco.mjtGeom.mjGEOM_LINE, a, [2, 0, 0], [.2, .7, .75, 1], b)
    elbow, tool = demo.ghost_points()
    shoulder = np.array([0, 0, config["base_height_m"]])
    ghost = [.15, .9, .3, .55]
    for point in (elbow, tool):
        add_marker(scene, mujoco.mjtGeom.mjGEOM_SPHERE, point, [.014, 0, 0], ghost)
    add_marker(scene, mujoco.mjtGeom.mjGEOM_CAPSULE, shoulder, [.004, 0, 0], ghost, elbow)
    add_marker(scene, mujoco.mjtGeom.mjGEOM_CAPSULE, elbow, [.004, 0, 0], ghost, tool)


def run_viewer(duration=None):
    """Open MuJoCo's native viewer; duration is optional for timed checks."""
    demo = DynamicsDemo()
    instructions = ("F8 mode | F9 select joint | Left - | Right + | F10 zero efforts\n"
                    "Down gravity hold | F11 run-pause | F12 reset | Close window to return\n"
                    "Green ghost = our equations; robot = MuJoCo")
    with mujoco.viewer.launch_passive(demo.model, demo.data, key_callback=demo.on_key,
                                      show_left_ui=False, show_right_ui=False) as viewer:
        with viewer.lock():
            viewer.cam.lookat[:] = [.12, 0, .17]
            viewer.cam.distance = 1.0
            viewer.cam.azimuth = 115
            viewer.cam.elevation = -30
            viewer.opt.sitegroup[:] = 0
        started = previous = time.perf_counter()
        while viewer.is_running():
            now = time.perf_counter()
            with viewer.lock():
                while True:
                    try:
                        demo.handle(demo.commands.get_nowait())
                    except Empty:
                        break
                demo.tick(min(now - previous, .04))
                update_markers(demo, viewer.user_scn)
                readout = demo.overlay()
            previous = now
            viewer.set_texts([
                (mujoco.mjtFontScale.mjFONTSCALE_100, mujoco.mjtGridPos.mjGRID_TOPLEFT, readout, ""),
                (mujoco.mjtFontScale.mjFONTSCALE_100, mujoco.mjtGridPos.mjGRID_BOTTOMLEFT, instructions, ""),
            ])
            viewer.sync()
            if duration is not None and now - started >= duration:
                break
            time.sleep(max(0, 1 / 60 - (time.perf_counter() - now)))
    return demo

Run this cell to open the viewer. It starts in TORQUE mode with the gravity hold on, so nothing moves until you press F11. Then try:

1. Press F11. The shoulder is already selected, so tap Right twice (0.1 Nm). The arm accelerates, and the elbow swings the opposite way because of the coupling term $d_{12}$. The ghost stays on the robot.
2. Press Down to turn off the gravity hold. The carriage drops to its lower limit, and the overlay reports the limit force.
3. Press F8 for TRAJECTORY mode and F11 to run. The computed torques make the robot follow the planned path with no feedback.

Close the window before restarting this cell or the kernel.

In [ ]:
demo = run_viewer()
print("Viewer closed. The notebook can continue.")